# Art → Real World (Office-Home) — I2I Translation & UDA

**Dataset:** Office-Home (65 classes)  
**Source domain:** Art  
**Target domain:** Real World  

This notebook runs:
- **Task I:** Spatial CycleGAN vs Spectral CycleGAN style transfer comparison
- **Task II:** 5 UDA methods benchmark (Source-only, CycleGAN, Spectral CycleGAN, CyCADA, FDA)

## 1. Setup

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# === Quick Test Mode ===
# Set FAST_TEST = True to verify the notebook runs end-to-end (~5 min)
# Set FAST_TEST = False for full experiment
FAST_TEST = True

if FAST_TEST:
    CYCLEGAN_EPOCHS = 1
    CYCLEGAN_DECAY = 0
    CYCADA_EPOCHS = 1
    CLASSIFIER_EPOCHS = 2
    MAX_IMAGES = 100
    NUM_TEST = 50
else:
    CYCLEGAN_EPOCHS = 50
    CYCLEGAN_DECAY = 50
    CYCADA_EPOCHS = 50
    CLASSIFIER_EPOCHS = 20
    MAX_IMAGES = 5000
    NUM_TEST = 60000

print(f'FAST_TEST={FAST_TEST}: epochs={CYCLEGAN_EPOCHS}+{CYCLEGAN_DECAY}, images={MAX_IMAGES}')

In [ ]:
# Clone CycleGAN repo
import os
if not os.path.exists('pytorch-CycleGAN-and-pix2pix'):
    !git clone https://github.com/junyanz/pytorch-CycleGAN-and-pix2pix.git
    !pip install -q visdom dominate


In [ ]:
import sys
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets, transforms
from torchvision.utils import save_image
from PIL import Image
import shutil
import glob

# Add utils to path
UTILS_DIR = '/content/drive/MyDrive/Project2/utils'
sys.path.insert(0, os.path.dirname(UTILS_DIR))

from utils.data_utils import get_office_home, get_unnormalized_transform, get_paired_loaders
from utils.classifiers import ResNet50Classifier, train_classifier, evaluate_classifier
from utils.cyclegan_wrapper import prepare_cyclegan_data, get_cyclegan_train_cmd, get_cyclegan_test_cmd
from utils.fft_utils import fda_transfer, spectral_decompose, spectral_reconstruct
from utils.spectral_cyclegan import save_low_freq_images, reconstruct_from_translated_low
from utils.cycada import train_cycada, translate_with_cycada
from utils.eval_utils import save_results
from utils.viz_utils import show_image_grid, print_results_table

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

In [ ]:
# Paths
DATA_ROOT = '/content/drive/MyDrive/Project2/data/office_home'
SAVE_DIR = '/content/drive/MyDrive/Project2/results/art_realworld'
CYCLEGAN_DATA = '/content/cyclegan_data/art2realworld'
os.makedirs(SAVE_DIR, exist_ok=True)

# Constants
NUM_CLASSES = 65
IMG_SIZE = 224
BATCH_SIZE = 32

# CycleGAN params
LOAD_SIZE = 256
CROP_SIZE = 224
INPUT_NC = 3
OUTPUT_NC = 3
NET_G = 'resnet_9blocks'

In [ ]:
# Office-Home dataset — already downloaded and placed in Drive
import os
assert os.path.exists('/content/drive/MyDrive/Project2/data/office_home/Art'), \
    "Office-Home not found! Please upload to Drive: MyDrive/Project2/data/office_home/{Art,Real World,...}"
print("Office-Home dataset found.")

## 2. Load Data

In [ ]:
# Normalized datasets (for classifier training/evaluation)
source_dataset = get_office_home(root=DATA_ROOT, domain='Art', img_size=IMG_SIZE)
target_dataset = get_office_home(root=DATA_ROOT, domain='Real World', img_size=IMG_SIZE)

print(f'Source (Art): {len(source_dataset)} images, {NUM_CLASSES} classes')
print(f'Target (Real World): {len(target_dataset)} images, {NUM_CLASSES} classes')

source_loader = DataLoader(source_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, drop_last=True)
target_loader = DataLoader(target_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, drop_last=True)

In [ ]:
# Unnormalized datasets (for CycleGAN / CyCADA / FDA - images in [0,1])
unnorm_transform = get_unnormalized_transform(img_size=IMG_SIZE)

source_path = os.path.join(DATA_ROOT, 'Art')
target_path = os.path.join(DATA_ROOT, 'Real World')

source_unnorm = datasets.ImageFolder(source_path, transform=unnorm_transform)
target_unnorm = datasets.ImageFolder(target_path, transform=unnorm_transform)

source_unnorm_loader = DataLoader(source_unnorm, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, drop_last=True)
target_unnorm_loader = DataLoader(target_unnorm, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, drop_last=True)

print(f'Unnormalized source: {len(source_unnorm)} images')
print(f'Unnormalized target: {len(target_unnorm)} images')

## Task I: Style Transfer Comparison

### 3a. Save source/target images for CycleGAN training

In [ ]:
# Save all source and target images as flat directories for CycleGAN
src_flat_dir = os.path.join(CYCLEGAN_DATA, 'flat_source')
tgt_flat_dir = os.path.join(CYCLEGAN_DATA, 'flat_target')
os.makedirs(src_flat_dir, exist_ok=True)
os.makedirs(tgt_flat_dir, exist_ok=True)

def flatten_imagefolder(root_dir, output_dir):
    """Copy all images from class subdirectories into a flat directory."""
    count = 0
    for class_name in sorted(os.listdir(root_dir)):
        class_dir = os.path.join(root_dir, class_name)
        if not os.path.isdir(class_dir):
            continue
        for fname in os.listdir(class_dir):
            if fname.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
                src_path = os.path.join(class_dir, fname)
                # Prefix with class name to avoid filename collisions
                dst_path = os.path.join(output_dir, f'{class_name}_{fname}')
                if not os.path.exists(dst_path):
                    shutil.copy2(src_path, dst_path)
                count += 1
    print(f'Copied {count} images to {output_dir}')

flatten_imagefolder(source_path, src_flat_dir)
flatten_imagefolder(target_path, tgt_flat_dir)

In [ ]:
# Prepare CycleGAN directory structure (trainA=source, trainB=target)
prepare_cyclegan_data(src_flat_dir, tgt_flat_dir, CYCLEGAN_DATA)

### 3b. Train Spatial CycleGAN (Art → Real World)

In [ ]:
# Train standard (spatial) CycleGAN
spatial_cmd = get_cyclegan_train_cmd(
    n_epochs=CYCLEGAN_EPOCHS,
    n_epochs_decay=CYCLEGAN_DECAY,
    dataroot=CYCLEGAN_DATA,
    name='art2realworld_spatial',
    load_size=LOAD_SIZE,
    crop_size=CROP_SIZE,
    input_nc=INPUT_NC,
    output_nc=OUTPUT_NC,
    netG=NET_G,
)
print(spatial_cmd)

In [ ]:
!{spatial_cmd}

### 3c. Train Spectral CycleGAN

Spectral CycleGAN decomposes images into low-freq and high-freq via FFT. CycleGAN is trained only on the low-frequency component. At inference, translated low-freq is recombined with the original high-freq.

In [ ]:
# Decompose source and target images into low/high frequency components
spectral_src_dir = os.path.join(CYCLEGAN_DATA, 'spectral_source')
spectral_tgt_dir = os.path.join(CYCLEGAN_DATA, 'spectral_target')

src_low_dir, src_high_dir = save_low_freq_images(src_flat_dir, spectral_src_dir, beta=0.03)
tgt_low_dir, tgt_high_dir = save_low_freq_images(tgt_flat_dir, spectral_tgt_dir, beta=0.03)

In [ ]:
# Prepare CycleGAN data for spectral (low-freq only)
spectral_cyclegan_data = os.path.join(CYCLEGAN_DATA, 'spectral')
prepare_cyclegan_data(src_low_dir, tgt_low_dir, spectral_cyclegan_data)

In [ ]:
# Train Spectral CycleGAN (on low-freq images only)
spectral_cmd = get_cyclegan_train_cmd(
    n_epochs=CYCLEGAN_EPOCHS,
    n_epochs_decay=CYCLEGAN_DECAY,
    dataroot=spectral_cyclegan_data,
    name='art2realworld_spectral',
    load_size=LOAD_SIZE,
    crop_size=CROP_SIZE,
    input_nc=INPUT_NC,
    output_nc=OUTPUT_NC,
    netG=NET_G,
)
print(spectral_cmd)

In [ ]:
!{spectral_cmd}

### 3d. Generate translated images with both CycleGANs

In [ ]:
import glob, shutil, os

# Prepare testA/testB for CycleGAN test
cyclegan_base = '/content/cyclegan_data/art2realworld'
spectral_base = os.path.join(cyclegan_base, 'spectral')

for base in [cyclegan_base, spectral_base]:
    for split in ['trainA', 'trainB']:
        src_dir = os.path.join(base, split)
        dst_dir = os.path.join(base, split.replace('train', 'test'))
        os.makedirs(dst_dir, exist_ok=True)
        if os.path.exists(src_dir):
            for f in sorted(glob.glob(f'{src_dir}/*.png') + glob.glob(f'{src_dir}/*.jpg'))[:50]:
                shutil.copy2(f, dst_dir)
            print(f"Copied {len(os.listdir(dst_dir))} images to {dst_dir}")

In [ ]:
# Generate spatial CycleGAN translations
spatial_test_cmd = get_cyclegan_test_cmd(
    netG='resnet_9blocks',
    dataroot=CYCLEGAN_DATA,
    name='art2realworld_spatial',
    load_size=LOAD_SIZE,
    crop_size=CROP_SIZE,
    input_nc=INPUT_NC,
    output_nc=OUTPUT_NC,
)
print(spatial_test_cmd)
!{spatial_test_cmd}

In [ ]:
# Generate spectral CycleGAN translations (on low-freq images)
spectral_test_cmd = get_cyclegan_test_cmd(
    netG='resnet_9blocks',
    dataroot=spectral_cyclegan_data,
    name='art2realworld_spectral',
    load_size=LOAD_SIZE,
    crop_size=CROP_SIZE,
    input_nc=INPUT_NC,
    output_nc=OUTPUT_NC,
)
print(spectral_test_cmd)
!{spectral_test_cmd}

In [ ]:
# Reconstruct spectral CycleGAN outputs: translated_low + original_high
translated_low_dir = os.path.join('results', 'art2realworld_spectral', 'test_latest', 'images')
# Filter to only fake_B images (source->target translations)
spectral_recon_dir = os.path.join(CYCLEGAN_DATA, 'spectral_reconstructed')

# The CycleGAN test output has *_fake_B.png files (A->B translations)
# We need to match them with original high-freq components
os.makedirs(spectral_recon_dir, exist_ok=True)

to_tensor = transforms.ToTensor()
fake_b_files = sorted(glob.glob(os.path.join(translated_low_dir, '*_fake_B.png')))
print(f'Found {len(fake_b_files)} translated low-freq images')

# Build a lookup of high-freq files by stem (without extension)
high_freq_by_stem = {}
for f in os.listdir(src_high_dir):
    stem = os.path.splitext(f)[0]
    high_freq_by_stem[stem] = f

for fpath in fake_b_files:
    fname = os.path.basename(fpath)
    # Extract original filename stem: CycleGAN names files as XXXX_fake_B.png
    base_stem = fname.replace('_fake_B.png', '')
    
    # Try matching by stem regardless of extension (.png, .jpg, etc.)
    high_fname = high_freq_by_stem.get(base_stem)
    if high_fname:
        high_path = os.path.join(src_high_dir, high_fname)
        low_img = to_tensor(Image.open(fpath).convert('RGB')).unsqueeze(0)
        high_img = to_tensor(Image.open(high_path).convert('RGB')).unsqueeze(0)
        recon = spectral_reconstruct(low_img, high_img)
        save_image(recon.squeeze(0), os.path.join(spectral_recon_dir, high_fname))

print(f'Reconstructed {len(os.listdir(spectral_recon_dir))} spectral CycleGAN images')

### 3e. Visualize: Spatial vs Spectral CycleGAN

In [ ]:
# Load sample images for visualization
def load_sample_images(image_dir, n=8, pattern='*.png'):
    """Load first n images from a directory as a tensor."""
    files = sorted(glob.glob(os.path.join(image_dir, pattern)))[:n]
    imgs = []
    to_t = transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.ToTensor()])
    for f in files:
        img = to_t(Image.open(f).convert('RGB'))
        imgs.append(img)
    return torch.stack(imgs) if imgs else torch.zeros(1, 3, IMG_SIZE, IMG_SIZE)

# Source images
src_samples = load_sample_images(src_flat_dir, n=8, pattern='*.jpg')
if len(glob.glob(os.path.join(src_flat_dir, '*.jpg'))) == 0:
    src_samples = load_sample_images(src_flat_dir, n=8, pattern='*.*')

# Target images
tgt_samples = load_sample_images(tgt_flat_dir, n=8, pattern='*.jpg')
if len(glob.glob(os.path.join(tgt_flat_dir, '*.jpg'))) == 0:
    tgt_samples = load_sample_images(tgt_flat_dir, n=8, pattern='*.*')

# Spatial CycleGAN output
spatial_out_dir = os.path.join('results', 'art2realworld_spatial', 'test_latest', 'images')
spatial_samples = load_sample_images(spatial_out_dir, n=8, pattern='*_fake_B.png')

# Spectral CycleGAN reconstructed output
spectral_samples = load_sample_images(spectral_recon_dir, n=8)

show_image_grid(
    {
        'Source (Art)': src_samples,
        'Target (Real World)': tgt_samples,
        'Spatial CycleGAN': spatial_samples,
        'Spectral CycleGAN': spectral_samples,
    },
    nrow=8,
    figsize=(16, 4),
    title='Task I: Art \u2192 Real World Style Transfer Comparison',
    save_path=os.path.join(SAVE_DIR, 'task1_comparison.png'),
)

---
## Task II: UDA Benchmark

### 4a. Source-Only Baseline

In [ ]:
# Helper: ImageNet normalization transform (applied on [0,1] tensors)
imagenet_normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

def normalize_batch(images):
    """Apply ImageNet normalization to a batch of [0,1] images."""
    return torch.stack([imagenet_normalize(img) for img in images])

In [ ]:
# Train classifier on source domain (Art) with ImageNet normalization
results = {}

model_source = ResNet50Classifier(num_classes=NUM_CLASSES)
model_source = train_classifier(model_source, source_loader, epochs=CLASSIFIER_EPOCHS, lr=1e-3, device=device)

# Evaluate on target domain (Real World)
print('\n--- Source-Only Evaluation ---')
acc_source_only = evaluate_classifier(model_source, target_loader, device=device)
results['Source-Only'] = acc_source_only

### 4b. CycleGAN UDA

Translate source (Art) images to target (Real World) style using trained spatial CycleGAN, then train classifier on translated images.

In [ ]:
# Build translated source dataset from CycleGAN outputs
# CycleGAN test outputs fake_B images (A->B = source->target style)
cyclegan_translated_dir = os.path.join('results', 'art2realworld_spatial', 'test_latest', 'images')

def build_translated_loader(translated_dir, source_dataset, batch_size, pattern='*_fake_B.png'):
    """
    Build a DataLoader from translated images + original source labels.
    Images are loaded in [0,1], then ImageNet-normalized for classifier.
    """
    translated_files = sorted(glob.glob(os.path.join(translated_dir, pattern)))
    to_t = transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.ToTensor()])
    
    all_images = []
    all_labels = []
    
    for fpath in translated_files:
        img = to_t(Image.open(fpath).convert('RGB'))
        img = imagenet_normalize(img)
        all_images.append(img)
    
    # Match labels from source dataset (same ordering)
    # CycleGAN preserves the order from trainA
    for i in range(min(len(all_images), len(source_dataset))):
        _, label = source_dataset[i]
        all_labels.append(label)
    
    n = min(len(all_images), len(all_labels))
    all_images = torch.stack(all_images[:n])
    all_labels = torch.tensor(all_labels[:n])
    
    dataset = TensorDataset(all_images, all_labels)
    return DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=0)

cyclegan_loader = build_translated_loader(
    cyclegan_translated_dir, source_unnorm, BATCH_SIZE
)
print(f'CycleGAN translated dataset: {len(cyclegan_loader.dataset)} images')

In [ ]:
# Train classifier on CycleGAN-translated source images
model_cyclegan = ResNet50Classifier(num_classes=NUM_CLASSES)
model_cyclegan = train_classifier(model_cyclegan, cyclegan_loader, epochs=CLASSIFIER_EPOCHS, lr=1e-3, device=device)

print('\n--- CycleGAN UDA Evaluation ---')
acc_cyclegan = evaluate_classifier(model_cyclegan, target_loader, device=device)
results['CycleGAN UDA'] = acc_cyclegan

### 4c. Spectral CycleGAN UDA

Same as above but using Spectral CycleGAN translations (translated low-freq + original high-freq).

In [ ]:
# Build translated loader from spectral CycleGAN reconstructed images
spectral_loader = build_translated_loader(
    spectral_recon_dir, source_unnorm, BATCH_SIZE, pattern='*.png'
)
print(f'Spectral CycleGAN translated dataset: {len(spectral_loader.dataset)} images')

In [ ]:
# Train classifier on Spectral CycleGAN-translated source images
model_spectral = ResNet50Classifier(num_classes=NUM_CLASSES)
model_spectral = train_classifier(model_spectral, spectral_loader, epochs=CLASSIFIER_EPOCHS, lr=1e-3, device=device)

print('\n--- Spectral CycleGAN UDA Evaluation ---')
acc_spectral = evaluate_classifier(model_spectral, target_loader, device=device)
results['Spectral CycleGAN UDA'] = acc_spectral

### 4d. CyCADA

CycleGAN + task loss: jointly trains image translation and ensures the classifier's predictions are preserved on translated images.

In [ ]:
# Train a source classifier on unnormalized images for CyCADA task loss
# CyCADA internally works on [0,1] images, so the task classifier must accept [0,1]
model_source_unnorm = ResNet50Classifier(num_classes=NUM_CLASSES)
model_source_unnorm = train_classifier(
    model_source_unnorm, source_unnorm_loader, epochs=CLASSIFIER_EPOCHS, lr=1e-3, device=device
)

In [ ]:
# Train CyCADA
G_s2t, G_t2s = train_cycada(
    source_loader=source_unnorm_loader,
    target_loader=target_unnorm_loader,
    classifier=model_source_unnorm,
    num_classes=NUM_CLASSES,
    in_channels=3,
    n_epochs=CYCADA_EPOCHS,
    lr=2e-4,
    lambda_cyc=10.0,
    lambda_task=1.0,
    device=device,
)

In [ ]:
# Translate source images to target style using CyCADA generator
cycada_translated = translate_with_cycada(G_s2t, source_unnorm_loader, device=device)

# Build normalized dataset for classifier training
cycada_images = []
cycada_labels = []
for imgs, labels in cycada_translated:
    cycada_images.append(normalize_batch(imgs))
    cycada_labels.append(labels)

cycada_images = torch.cat(cycada_images, dim=0)
cycada_labels = torch.cat(cycada_labels, dim=0)

cycada_dataset = TensorDataset(cycada_images, cycada_labels)
cycada_loader = DataLoader(cycada_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
print(f'CyCADA translated dataset: {len(cycada_dataset)} images')

In [ ]:
# Train classifier on CyCADA-translated source images
model_cycada = ResNet50Classifier(num_classes=NUM_CLASSES)
model_cycada = train_classifier(model_cycada, cycada_loader, epochs=CLASSIFIER_EPOCHS, lr=1e-3, device=device)

print('\n--- CyCADA Evaluation ---')
acc_cycada = evaluate_classifier(model_cycada, target_loader, device=device)
results['CyCADA'] = acc_cycada

### 4e. FDA (Fourier Domain Adaptation)

No GAN training needed. Swap low-frequency FFT bands from target into source images.

In [ ]:
# Apply FDA: replace source low-freq amplitude with target's
fda_images = []
fda_labels = []
beta = 0.01  # Low-freq bandwidth ratio

target_iter = iter(target_unnorm_loader)

for src_imgs, src_labels in source_unnorm_loader:
    try:
        tgt_imgs, _ = next(target_iter)
    except StopIteration:
        target_iter = iter(target_unnorm_loader)
        tgt_imgs, _ = next(target_iter)
    
    # Match batch sizes
    n = min(src_imgs.size(0), tgt_imgs.size(0))
    src_imgs = src_imgs[:n]
    tgt_imgs = tgt_imgs[:n]
    src_labels = src_labels[:n]
    
    # FDA transfer (works on [0,1] images)
    transferred = fda_transfer(src_imgs, tgt_imgs, beta=beta)
    
    # Normalize for classifier
    transferred_norm = normalize_batch(transferred)
    fda_images.append(transferred_norm)
    fda_labels.append(src_labels)

fda_images = torch.cat(fda_images, dim=0)
fda_labels = torch.cat(fda_labels, dim=0)

fda_dataset = TensorDataset(fda_images, fda_labels)
fda_loader = DataLoader(fda_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
print(f'FDA dataset: {len(fda_dataset)} images')

In [ ]:
# Train classifier on FDA-transferred source images
model_fda = ResNet50Classifier(num_classes=NUM_CLASSES)
model_fda = train_classifier(model_fda, fda_loader, epochs=CLASSIFIER_EPOCHS, lr=1e-3, device=device)

print('\n--- FDA Evaluation ---')
acc_fda = evaluate_classifier(model_fda, target_loader, device=device)
results['FDA'] = acc_fda

### 4f. Results Summary

In [ ]:
# Print results table
print_results_table(results, 'Art \u2192 Real World (Office-Home)')

In [ ]:
# Save results to Drive
save_results(results, 'Art\u2192Real World', save_dir=SAVE_DIR)

In [ ]:
# Visualize UDA translations comparison
# Collect sample translated images from each method for visual comparison

# Source samples (unnormalized for display)
src_vis, _ = next(iter(source_unnorm_loader))
src_vis = src_vis[:8]

# Target samples
tgt_vis, _ = next(iter(target_unnorm_loader))
tgt_vis = tgt_vis[:8]

# CyCADA translations
cycada_vis = cycada_translated[0][0][:8] if cycada_translated else src_vis

# FDA translations (unnormalized for display)
target_iter_vis = iter(target_unnorm_loader)
tgt_for_fda, _ = next(target_iter_vis)
n = min(src_vis.size(0), tgt_for_fda.size(0))
fda_vis = fda_transfer(src_vis[:n], tgt_for_fda[:n], beta=beta)

show_image_grid(
    {
        'Source (Art)': src_vis,
        'Target (Real World)': tgt_vis,
        'CyCADA': cycada_vis,
        'FDA': fda_vis,
    },
    nrow=8,
    figsize=(16, 4),
    title='Task II: Art \u2192 Real World UDA Translations',
    save_path=os.path.join(SAVE_DIR, 'task2_uda_comparison.png'),
)

In [ ]:
# Bar chart of results
methods = list(results.keys())
accuracies = [results[m] for m in methods]

plt.figure(figsize=(10, 5))
bars = plt.bar(methods, accuracies, color=['#4C72B0', '#55A868', '#C44E52', '#8172B2', '#CCB974'])
plt.ylabel('Target Accuracy')
plt.title('Art \u2192 Real World (Office-Home) \u2014 UDA Results')
plt.ylim(0, 1.0)
for bar, acc in zip(bars, accuracies):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{acc:.2%}', ha='center', fontsize=10)
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'task2_results_bar.png'), dpi=150, bbox_inches='tight')
plt.show()

print('All results saved to:', SAVE_DIR)